In [ ]:
from pathlib import Path
from IPython.display import Image, SVG, display
import getpass
import os
import re
import time

# Works when this notebook starts in notebooks/, the repository root, or Gadi.
candidates = [Path.cwd(), *Path.cwd().parents,
              Path(os.path.expandvars("/scratch/vp91/$USER/intro-to-cuda"))]
repo_root = next((path for path in candidates
                  if (path / "exercises/CMakeLists.txt").is_file()
                  and (path / "job-script").is_dir()), None)
if repo_root is None:
    raise FileNotFoundError("Open this notebook from the intro-to-cuda checkout.")
os.chdir(repo_root)
job_dir = repo_root / "job-script"
username = getpass.getuser()


def show_job_output(job_name, job_id):
    job_number = job_id.split(".")[0]
    log = job_dir / f"{job_name}.o{job_number}"
    if log.is_file():
        print(log.read_text())
    else:
        print(f"Output is not available yet: {log.name}. Check qstat, then rerun this cell.")


def outputs_ready(paths, submitted_at):
    if all(path.is_file() and path.stat().st_mtime >= submitted_at for path in paths):
        return True
    print("New results are not available yet. Wait for this job to finish, then rerun this cell.")
    return False

print(f"Repository: {repo_root}")

# CUDA Events


> **Overview**
>
> 
>  * **Time:** 30 min
> 
>  1. Learn about CUDA events and their usage.
>  1. Understand how to measure elapsed time using CUDA events.

CUDA events are a powerful feature that allows you to measure the time taken by operations on the GPU.
They can be used to synchronize between different streams and to profile the performance of your CUDA
applications. Events are lightweight and provide a way to track the completion of operations without
blocking the CPU.

## CUDA Event Basics

In [ ]:
# CUDA Events
display(Image(filename=str(repo_root / "docs/source/tutorial/figs/cuda-events.png")))

CUDA events are used to mark points in time in your CUDA application. They can be created, recorded, and
queried to determine when certain operations have completed. Events can be used to measure the elapsed time
between two points in your code, which is useful for performance profiling. CUDA events are created using
the `cudaEventCreate` function, and you can record an event using `cudaEventRecord`.

```cpp
cudaEvent_t start, stop;
cudaEventCreate(&start);
cudaEventCreate(&stop);

// Record the start event
cudaEventRecord(start, 0);

// Perform some GPU operations here

// Record the stop event
cudaEventRecord(stop, 0);

// Wait for the stop event to complete
cudaEventSynchronize(stop);

float milliseconds = 0;
cudaEventElapsedTime(&milliseconds, start, stop);
printf("Elapsed time: %f ms\n", milliseconds);
```

### Timing a kernel on the default stream

Record the start event, launch the kernel, then record the stop event on
stream 0. The stop timestamp is recorded after the kernel finishes.
`cudaEventElapsedTime` measures the time between the two event timestamps.


In [ ]:
from IPython.display import HTML, display

display(HTML("""
<style>
.cuda-event-timing { max-width: 640px; }
.cuda-event-timing svg { display: block; width: 100%; height: auto; }
.cuda-event-timing .event-cursor { animation: cuda-event-sweep 7s linear infinite; }
.cuda-event-timing input:checked ~ svg .event-cursor { animation-play-state: paused; }
@keyframes cuda-event-sweep { to { transform: translateX(500px); } }
@media (prefers-reduced-motion: reduce) {
    .cuda-event-timing .event-cursor { display: none; }
}
</style>
<div class="cuda-event-timing">
<input type="checkbox" id="cuda-event-timing-pause">
<label for="cuda-event-timing-pause">Pause cursor</label>
<svg viewBox="0 0 640 155" role="img" aria-labelledby="cuda-event-timing-title">
  <title id="cuda-event-timing-title">Record start, launch the kernel, then record stop on the default stream</title>
  <rect width="640" height="155" rx="8" fill="#f8fafc"/>
  <g font-family="system-ui, sans-serif" font-size="13" fill="#172033" text-anchor="middle">
    <text x="100" y="32" font-weight="700">1. Record start</text>
    <text x="320" y="32" font-weight="700">2. Launch kernel</text>
    <text x="540" y="32" font-weight="700">3. Record stop</text>
    <rect x="100" y="41" width="440" height="65" rx="5" fill="#dbeafe"/>
    <path d="M100 41 V106 M540 41 V106" stroke="#64748b" stroke-dasharray="3 3"/>
    <path d="M65 65 H575 l-6 -4 m6 4 l-6 4" fill="none" stroke="#64748b"/>
    <rect x="200" y="51" width="240" height="28" rx="5" fill="#6d28d9"/>
    <text x="320" y="70" fill="white">kernel/D2H/H2D/D2D</text>
    <rect x="66" y="54" width="68" height="22" rx="4" fill="#facc15"/>
    <text x="100" y="69" font-size="11">Start event</text>
    <rect x="506" y="54" width="68" height="22" rx="4" fill="#facc15"/>
    <text x="540" y="69" font-size="11">Stop event</text>
    <path d="M100 96 v9 H540 v-9" fill="none" stroke="#475569"/>
    <text x="320" y="126" font-weight="700">Elapsed time window</text>
    <text x="320" y="143" font-size="11">Stop timestamp − start timestamp</text>
  </g>
  <g class="event-cursor" aria-hidden="true">
    <path d="M70 42 V89" stroke="white" stroke-width="4"/>
    <path d="M70 42 V89" stroke="#111827" stroke-width="2" stroke-dasharray="4 3"/>
  </g>
</svg>
</div>
"""))


## Cross stream Synchronization


CUDA events can also be used for cross-stream synchronization. When you record an event in one stream,
it can be waited on in another stream. This allows you to synchronize operations across different streams without blocking the CPU.

```cpp
cudaStream_t stream1, stream2;
cudaStreamCreate(&stream1);
cudaStreamCreate(&stream2);

cudaEvent_t event;
cudaEventCreate(&event);

// Perform the producer operation in stream1
kernel1<<<blocks, threads, 0, stream1>>>(...);

// Capture completion of the producer, then make stream2 wait for it
cudaEventRecord(event, stream1);

// Wait for the event in stream2
cudaStreamWaitEvent(stream2, event, 0);

// Perform some operations in stream2 that depend on the completion of stream1
kernel2<<<blocks, threads, 0, stream2>>>(...);

// Wait before releasing resources used by the consumer
cudaStreamSynchronize(stream2);

// Cleanup
cudaEventDestroy(event);
cudaStreamDestroy(stream1);
cudaStreamDestroy(stream2);
```

> **Explanation**
>
> 
>  `cudaStreamWaitEvent` is used to make a stream wait for an event recorded in another stream.

## Timing events across streams


When timing events across streams, you can record events in one stream and then wait for those events
in another stream. This allows you to measure the time taken by operations in different streams without
blocking the CPU.

```cpp
cudaStream_t stream1, stream2;
cudaStreamCreate(&stream1);
cudaStreamCreate(&stream2);

cudaEvent_t start, stop, producerDone;
cudaEventCreate(&start);
cudaEventCreate(&stop);
cudaEventCreate(&producerDone);

// Record the start event in stream1
cudaEventRecord(start, stream1);

// Perform some operations in stream1
kernel1<<<blocks, threads, 0, stream1>>>(...);

// Make the stop marker depend on completion of kernel1
cudaEventRecord(producerDone, stream1);
cudaStreamWaitEvent(stream2, producerDone, 0);
cudaEventRecord(stop, stream2);

// This host-side wait ensures both timestamps are available
cudaEventSynchronize(stop);

float milliseconds = 0;
cudaEventElapsedTime(&milliseconds, start, stop);
printf("Elapsed time across streams: %f ms\n", milliseconds);

cudaEventDestroy(start);
cudaEventDestroy(stop);
cudaEventDestroy(producerDone);
cudaStreamDestroy(stream1);
cudaStreamDestroy(stream2);
```

> **Important**
>
> 1. You cannot use an event recorded in one stream to measure the execution time of operations that occur in another stream unless you enforce proper synchronization.
> 1. If streams are non-blocking and concurrent, incorrect usage may lead to race conditions or invalid timings.

> **Key Points**
>
> 
>  1. CUDA events are used to measure elapsed time and synchronize operations across streams.
>  1. Events can be created, recorded, and queried to determine the completion of operations.
>  1. Cross-stream synchronization is achieved using `cudaStreamWaitEvent`.
>  1. Timing events across streams requires careful synchronization to ensure accurate measurements.
>  1. Events are lightweight and do not block the CPU, making them suitable for performance profiling.

## Event ordering in these examples

The notebook corrects two ordering issues in the RST examples: the cross-stream dependency is recorded **after** the producer kernel, and the cross-stream stop event waits for producer completion. An event captures work already submitted to its stream; later submissions are not included. See [NVIDIA event documentation](https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__EVENT.html).

`cudaStreamWaitEvent` queues a device-side dependency; `cudaEventSynchronize` waits on the CPU. Recording events can be asynchronous, but explicit host synchronization blocks until completion.

The snippets above illustrate API ordering and contain omitted kernels and launch parameters (`...`), so they are not standalone programs. The executable example below is complete.

## Exercise 08: CUDA Events

**Time: 20 min**

Inspect the event markers in the supplied program, then submit its documented job.
[Exercise instructions](../docs/source/tutorial/exercises/08-events.rst).

Run the setup cell first. Build the exercises once in
[02-gpu.ipynb](02-gpu.ipynb) before submitting jobs from this notebook.
After editing CUDA source, return to that notebook to rebuild.

Submission cells use the same `qsub` commands as the exercise documentation,
from `job-script`, and immediately show `qstat`. A submitted job may still be queued.
Use the monitoring cell to check again; rerunning a submission cell creates another job.
Read the PBS output and inspect generated files after the job finishes.

<a href="../exercises/08-events/vector-add-async.cu" target="_blank" rel="noopener noreferrer">Open vector-add-async.cu</a>

In [ ]:
# Submit a new job, then inspect the queue.
submitted_at = time.time()
submission = !cd "{job_dir}" && qsub 08-events.pbs
print("\n".join(submission))
!qstat -u "{username}"

# Keep the returned job ID so later cells read this run's PBS log.
job_id = next((line.strip() for line in submission
                 if re.fullmatch(r"\d+(?:\.[A-Za-z0-9_.-]+)?", line.strip())), None)
if job_id is None:
    raise RuntimeError("qsub did not return a job ID. Check its output above.")


### Check the queue again

Run this cell to monitor jobs without submitting another one. Once your job is no
longer listed, rerun its output/visualisation cells. Inspect the PBS log for errors.

In [ ]:
!qstat -u "{username}"


In [ ]:
!qstat -u "{username}"
show_job_output("08-events", job_id)


### Interpret the timings

Check result verification reports `PASSED`. Each iteration records its start
before host-to-device copies and its stop after the device-to-host copy in the
same stream. The interval covers transfers and kernel work, not just the kernel.
Do all iterations take the same time? Repeat the job and compare variability.